# Laboratorio 2: Il gateway e il conto dei token

---

# Lab 2: The gateway and the token count

**Come si usa.** Esegui le celle dall'alto in basso (Maiusc+Invio). Serve un account Google gratuito
per usare Colab. Il testo completo del laboratorio e' sul sito: https://corso-rag.giuseppebosi.com/moduli/02-chiavi-gateway-e-quote/LAB/

---

**How to use it.** Run the cells from top to bottom (Shift+Enter). Colab needs a free Google account.
The full text of the lab is on the site: https://corso-rag.giuseppebosi.com/en/moduli/02-chiavi-gateway-e-quote/LAB/

In [ ]:
# Prepara l'ambiente / Set up the environment (about 1 minute)
import os, sys, subprocess
if sys.version_info[:2] not in [(3, 11), (3, 12)]:
    print("ATTENZIONE: questo corso e' provato con Python 3.11 o 3.12; hai "
          f"{sys.version_info.major}.{sys.version_info.minor}. "
          "WARNING: the course is tested on Python 3.11 or 3.12.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE, "rag-chatbot-course")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JoeBosi/rag-chatbot-course.git", REPO_DIR], check=True)
os.chdir(os.path.join(REPO_DIR, "corso"))
!pip install -q openai python-dotenv httpx PyYAML pypdf
print("Cartella di lavoro / working folder:", os.getcwd())

## La chiave gratuita / The free key

Questo laboratorio chiama un modello: serve una chiave **Groq**, gratuita e senza carta di credito
([CHIAVI.md](https://corso-rag.giuseppebosi.com/CHIAVI/) spiega come ottenerla in due minuti).
Poi in Colab: icona della **chiave** nella barra a sinistra, **Aggiungi nuovo segreto**, nome
`GROQ_API_KEY`, valore la tua chiave, e attiva **Accesso da notebook**. Se non la imposti, la cella
te la chiede al momento. La chiave non finisce mai nel notebook.

---

This lab calls a model, so you need a free **Groq** key, no credit card
([CHIAVI.md](https://corso-rag.giuseppebosi.com/en/CHIAVI/) shows how to get one in two minutes).
Then in Colab: the **key** icon in the left bar, **Add new secret**, name `GROQ_API_KEY`, value your
key, and switch on **Notebook access**. If you skip this, the cell asks for the key when you run it.
The key is never saved in the notebook.

In [ ]:
# La chiave Groq / The Groq key. Mai scritta nel notebook / never written in the notebook.
import os
from getpass import getpass

def leggi_chiave(nome="GROQ_API_KEY"):
    if os.environ.get(nome):
        return os.environ[nome]
    try:
        from google.colab import userdata          # Colab: icona della chiave, a sinistra
        return userdata.get(nome)
    except Exception:
        pass
    try:
        return getpass(f"Incolla {nome} / paste {nome}: ")
    except Exception:
        return ""

chiave = leggi_chiave()
if chiave:
    os.environ["GROQ_API_KEY"] = chiave
    os.environ["LLM_PROVIDER"] = "groq"
    print("Chiave caricata (non viene mostrata) / key loaded (not displayed).")
else:
    print("Nessuna chiave. I passaggi che chiamano il modello si fermeranno con "
          "il messaggio 'Manca GROQ_API_KEY'. Vedi la cella sopra.\n"
          "No key. Steps that call the model will stop with 'Manca GROQ_API_KEY'. See the cell above.")
![ -f .env ] || cp .env.example .env

Le celle con un'etichetta **[serve la chiave]** chiamano il modello. Senza chiave si fermano con
`Manca GROQ_API_KEY in .env`: non e' un guasto, inserisci la chiave come spiegato sopra e rilancia.
Le altre celle funzionano anche senza.

---

Cells labelled **[needs the key]** call the model. Without a key they stop with
`Manca GROQ_API_KEY in .env` (the message is Italian): this is not a fault, add the key as explained
above and run the cell again. The other cells work without it.

## Passo 1: installa e avvia il gateway

In Colab il gateway gira in un processo di sfondo, in un ambiente virtuale suo (come spiega la lezione). L'installazione richiede un paio di minuti.

---

## Step 1: install and start the gateway

In Colab the gateway runs as a background process in its own virtual environment (as the lesson explains). Installing takes a couple of minutes.

In [ ]:
!python -m venv .venv-gateway
!.venv-gateway/bin/pip install -q -r requirements-gateway.txt
!mkdir -p memoria/gateway
!cp moduli/02-chiavi-gateway-e-quote/starter/config.yaml memoria/gateway/

Avvio del gateway sul solo indirizzo `127.0.0.1` (non togliere `--host`). Il processo resta acceso finche' non lo fermi con l'ultima cella.

---

Start the gateway on `127.0.0.1` only (do not drop `--host`). The process stays on until you stop it in the last cell.

In [ ]:
import subprocess, time, os
GATEWAY = subprocess.Popen(
    [".venv-gateway/bin/litellm", "--config", "memoria/gateway/config.yaml",
     "--host", "127.0.0.1", "--port", "4000"],
    stdout=open("gateway.log", "w"), stderr=subprocess.STDOUT, env=dict(os.environ))
for _ in range(180):
    time.sleep(2)
    if "Uvicorn running on http://127.0.0.1:4000" in open("gateway.log").read():
        print("Gateway acceso su http://127.0.0.1:4000 / gateway is up")
        break
else:
    print(open("gateway.log").read()[-1500:])

Prova **[serve la chiave]**: deve tornare un JSON con `"content":"pronto"` o simile. Con `max_tokens` a 200 e non a 5: i modelli `gpt-oss` ragionano prima di rispondere.

---

Try it **[needs the key]**: it must return JSON with `"content":"pronto"` or similar. `max_tokens` is 200, not 5: `gpt-oss` models reason before answering.

In [ ]:
!curl -s http://localhost:4000/v1/chat/completions \
  -H "Authorization: Bearer sk-memoria-locale" \
  -H "Content-Type: application/json" \
  -d '{"model":"memoria-veloce","messages":[{"role":"user","content":"Rispondi solo: pronto"}],"max_tokens":200}' | head -c 400

## Passo 2: fai passare il tuo sistema dal gateway

In Colab non serve modificare `.env`: si impostano le variabili d'ambiente. Poi si rilancia il sistema del modulo 1, senza toccare il codice.

---

## Step 2: send your system through the gateway

In Colab you do not need to edit `.env`: set the environment variables. Then rerun the module 1 system without changing any code.

In [ ]:
os.environ.update(LLM_PROVIDER="custom", LLM_ENDPOINT="http://localhost:4000/v1",
                  LLM_MODEL="memoria-veloce", LLM_API_KEY="sk-memoria-locale")
!python memoria/memoria_minima.py "Cosa significa l'errore E4 sulla caldaia Termovex?"

## Passo 4: `stima()`

Sul corpus del corso deve dare esattamente 12 documenti, 17036 caratteri, 22 pezzi, 44 chiamate, tempo minimo 2.1 minuti. Non chiama il modello.

---

## Step 4: `stima()`

On the course corpus it must give exactly 12 documents, 17036 characters, 22 chunks, 44 calls, minimum time 2.1 minutes. It does not call the model.

## Starter o soluzione? / Starter or solution?

Nel laboratorio scrivi tu alcune funzioni (i TODO). Qui sotto scegli:
`USA_SOLUZIONE = True` copia la soluzione, cosi' tutte le celle girano subito;
`USA_SOLUZIONE = False` copia lo **starter** con i TODO: apri il file nel pannello **File** a sinistra
(cartella `rag-chatbot-course/corso/memoria`), scrivi la funzione, salva e rilancia le celle.

---

In the lab you write some functions yourself (the TODOs). Choose below:
`USA_SOLUZIONE = True` copies the solution so every cell runs straight away;
`USA_SOLUZIONE = False` copies the **starter** with the TODOs: open the file in the **Files** panel on
the left (folder `rag-chatbot-course/corso/memoria`), write the function, save, and run the cells again.

In [ ]:
USA_SOLUZIONE = True   # False = parti dallo starter e scrivi tu i TODO
MOD = "moduli/02-chiavi-gateway-e-quote"
ORIGINE = "soluzione" if USA_SOLUZIONE else "starter"
!mkdir -p memoria
!cp $MOD/$ORIGINE/stima_quota.py memoria/

In [ ]:
!python memoria/stima_quota.py progetto-finale/corpus/manuali/*.md progetto-finale/corpus/note/*.md

## Passo 5: output strutturato **[serve la chiave]**

Un JSON con apparecchio, componente, azione, data e i valori.

---

## Step 5: structured output **[needs the key]**

JSON with appliance, component, action, date and values.

In [ ]:
!python moduli/02-chiavi-gateway-e-quote/starter/estrai_json.py progetto-finale/corpus/note/2026-08-30-caldaia-manutenzione.md

## Controlla le chiavi **[serve la chiave]**

Prova ogni chiave presente e mostra la quota che resta.

---

## Check the keys **[needs the key]**

Tries every key present and shows the remaining quota.

In [ ]:
!python moduli/02-chiavi-gateway-e-quote/starter/controlla_chiavi.py

## Passo 6: verifica

Il gateway deve essere acceso (la cella di avvio sopra).

---

## Step 6: check

The gateway must be running (the start cell above).

In [ ]:
!python moduli/02-chiavi-gateway-e-quote/verifica.py

## Fine: ferma il gateway

---

## Done: stop the gateway

In [ ]:
GATEWAY.terminate(); print('gateway fermato / gateway stopped')